## Cell 1


In [ ]:
!pip install -q -U langchain langchain-google-genai langgraph

## Cell 2


In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass(
    "Enter your Gemini API key: "
)

## Cell 3


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0.3
)

## Cell 4


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a helpful research assistant.

        Explain information clearly and accurately.

        Adapt your explanation to the audience:
        {audience}

        Follow the requested task:
        {task}

        Use the supplied material as your primary source.
        """
    ),
    (
        "human",
        """
        Material:

        {text}
        """
    )
])

## Cell 5


In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = (
    prompt
    | model
    | StrOutputParser()
)

## Cell 6


In [ ]:
article = """
Inflation refers to a sustained increase in the
general price level of goods and services.

It can be caused by rising production costs,
increased consumer demand and expansion in
the money supply.
"""

response = chain.invoke({
    "audience": "a secondary school student",
    "task": "Summarise the material in simple language.",
    "text": article
})

print(response)

## Cell 7


In [ ]:
from langchain.tools import tool

@tool
def calculate_percentage(
    amount: float,
    percentage: float
) -> float:
    """
    Calculate a percentage of a given amount.

    Example:
    amount = 800
    percentage = 15

    Returns 120.
    """
    return amount * percentage / 100

## Cell 8


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[calculate_percentage],
    system_prompt="""
    You are a helpful research assistant.

    Answer general questions clearly.

    When the user asks to calculate a percentage
    of an amount, use the calculate_percentage tool.

    Explain the result in simple language.
    """
)

## Cell 9


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[],
    system_prompt="You are a helpful AI assistant."
)

## Cell 10


In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 15% of 800?"
        }
    ]
})

print(response["messages"][-1].content)

## Cell 11


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

## Cell 12


In [ ]:
chat_agent = create_agent(
    model=model,
    tools=[calculate_percentage],
    system_prompt="""
    You are a helpful research assistant.

    Remember the context of the current conversation.

    Answer questions clearly and adapt to the
    user's previous messages.

    Use the calculator tool when percentage
    calculations are requested.
    """,
    checkpointer=memory
)

## Cell 13


In [ ]:
config = {
    "configurable": {
        "thread_id": "research_session_1"
    }
}

response = chat_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "I am studying economics. Explain inflation simply."
            }
        ]
    },
    config=config
)

print(response["messages"][-1].content)

## Cell 14


In [ ]:
response = chat_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What are its three main causes?"
            }
        ]
    },
    config=config
)

print(response["messages"][-1].content)

## Cell 15


In [ ]:
def research_assistant(mode, text):

    if mode == "Summarise":

        return chain.invoke({
            "audience": "a general reader",
            "task": "Summarise the supplied material.",
            "text": text
        })

    elif mode == "Explain":

        return chain.invoke({
            "audience": "a secondary school student",
            "task": "Explain the supplied material simply.",
            "text": text
        })

    elif mode == "Calculate":

        result = chat_agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": text
                    }
                ]
            },
            config=config
        )

        return result["messages"][-1].content

    else:
        return "Invalid mode."

## Cell 16


In [ ]:
print(
    research_assistant(
        "Summarise",
        "Inflation is a sustained increase in the general price level."
    )
)